# Weather Silver Layer — Dedup & Transform

Reads from `workspace.bronze.weather_readings`, deduplicates by `event_id` (city|observation_time), casts types, and writes to `workspace.silver.weather_readings`.

Uses watermark on `polled_at` for dedup and `availableNow=True` trigger.

In [0]:
from pyspark.sql.functions import *

CHECKPOINT_LOCATION = "/Volumes/workspace/landing/checkpoints/weather/silver"
SILVER_TABLE = "workspace.silver.weather_readings"

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
# Table is auto-created by writeStream toTable with correct schema

In [0]:
bronze = spark.readStream.table("workspace.bronze.weather_readings")

silver = (bronze
    .drop("_rescued_data")
    .withColumn("observation_time", to_timestamp("observation_time_utc"))
    .withColumn("polled_at", to_timestamp("polled_at_utc"))
    .withColumn("temperature_2m", col("temperature_2m").cast("double"))
    .withColumn("relative_humidity_2m", col("relative_humidity_2m").cast("double"))
    .withColumn("apparent_temperature", col("apparent_temperature").cast("double"))
    .withColumn("precipitation", col("precipitation").cast("double"))
    .withColumn("rain", col("rain").cast("double"))
    .withColumn("weather_code", col("weather_code").cast("int"))
    .withColumn("cloud_cover", col("cloud_cover").cast("double"))
    .withColumn("pressure_msl", col("pressure_msl").cast("double"))
    .withColumn("wind_speed_10m", col("wind_speed_10m").cast("double"))
    .withColumn("wind_direction_10m", col("wind_direction_10m").cast("double"))
    .withColumn("wind_gusts_10m", col("wind_gusts_10m").cast("double"))
    .withColumn("is_day", col("is_day").cast("int"))
    .withWatermark("polled_at", "24 hours")
    .dropDuplicates(["event_id"]))

query = (silver.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(SILVER_TABLE))

query.awaitTermination()
print("✅ Silver stream completed — deduplicated & type-cast rows written")

In [0]:
row_count = spark.table(SILVER_TABLE).count()
dup_count = spark.table(SILVER_TABLE).groupBy("event_id").count().filter("count > 1").count()
print(f"📊 Silver table {SILVER_TABLE}: {row_count} rows, {dup_count} duplicate event_id groups")
spark.table(SILVER_TABLE).display()